In [ ]:
import itertools
from pathlib import Path
from typing import cast

import pandas as pd
from IPython.display import display

from utils.corr_assoc_utils import create_corr_assoc_matrix
from utils.misc_utils import contingency_table, display_side_by_side, turn_cats_to_continuous
from utils.saving_utils import DatasetBundle

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True


# def display_in_columns(df: pd.DataFrame, n_columns: int = 3):
#     display_side_by_side(
#         *(df.iloc[i * len(df) // n_columns : (i + 1) * len(df) // n_columns] for i in range(n_columns))
#     )

# Hardcoded paths, change to apply to new data
data = DatasetBundle.load(Path("data/preprocessed/data_full"))
data.meta = cast(pd.DataFrame, data.meta)
data.na = cast(pd.DataFrame, data.na)


# Define discovery and validation data split
discov_mask = data.data["zeit"].eq("2018")  # Discovery data is ESA 2018 and validation data is ESA 2021

### variable groups

In [ ]:
display_side_by_side(
    *(data.meta.loc[lambda x: x["group"].eq(y), ["name", "description"]] for y in data.meta["group"].cat.categories),
    captions=data.meta["group"].cat.categories,
)

### look at one variable

In [ ]:
var = "hazar"
display(data.meta[lambda x: x["name"].eq(var)])
display_side_by_side(
    pd.DataFrame(data.data[var].value_counts(dropna=False, normalize=False).sort_index()),
    pd.DataFrame(data.na[var].value_counts(dropna=False, normalize=False).sort_index()),
)

### search for variable information using regex

In [ ]:
data.meta[lambda x: x["name"].str.contains("hnb")]

### show how often two variables are equal and different

In [ ]:
# Check if two variables are equal (only works if both are continuous or categorical with the same labels)
vari, varj = "hnb30", "typ_hnb"

# True means vari is equal to varj, False means they are different
vars_equal = (data.data[vari] == data.data[varj]) | (data.data[vari].isna() & data.data[varj].isna()) & (
    data.na[vari] == data.na[varj]
)
vars_equal.value_counts(normalize=True)  # normalize=False for absolute values, normalize=True for percentages

In [ ]:
# Show instances where they are not equal
data.data.loc[~vars_equal, [vari, varj]]

In [ ]:
# Check if two categorical variables are the same
vari, varj = "hasch_12", "fr12_ha1"
# Use normalize="index" to show percentages for each index
contingency_table(data.data[vari], data.data[varj], dropna=False, normalize=False)

### variables with high missingness

In [ ]:
# Variables entirely missing from one dataset
not_discov_vars = data.na[discov_mask].loc[0:-1, lambda x: ~x.eq("observed").any()].columns
not_valid_vars = data.na[~discov_mask].loc[0:-1, lambda x: ~x.eq("observed").any()].columns
print("discovery all missing:  " + ", ".join(not_discov_vars))
print("validation all missing:  " + ", ".join(not_valid_vars))

In [ ]:
# Variables with highest missingness
discov_na = data.na[discov_mask].drop(columns=not_discov_vars)
valid_na = data.na[~discov_mask].drop(columns=not_valid_vars)
imputed_skip = ["observed", "imputed skip"]
rule_imputed = ["observed", "imputed skip", "rule imputed"]
mice_imputed = ["observed", "imputed skip", "rule imputed", "mice imputed"]
display_side_by_side(
    pd.DataFrame({"percent_observed": discov_na.isin(["observed"]).mean()[lambda x: x <= 0.5].sort_values()}),
    pd.DataFrame({"percent_observed": discov_na.isin(imputed_skip).mean()[lambda x: x <= 0.5].sort_values()}),
    pd.DataFrame({"percent_observed": discov_na.isin(rule_imputed).mean()[lambda x: x <= 0.5].sort_values()}),
    pd.DataFrame({"percent_observed": discov_na.isin(mice_imputed).mean()[lambda x: x <= 0.5].sort_values()}),
    captions=["discovery", "discovery with imputed skips", "discovery rule imputed", "discovery mice imputed"],
)
display_side_by_side(
    pd.DataFrame({"percent_observed": valid_na.isin(["observed"]).mean()[lambda x: x <= 0.5].sort_values()}),
    pd.DataFrame({"percent_observed": valid_na.isin(imputed_skip).mean()[lambda x: x <= 0.5].sort_values()}),
    pd.DataFrame({"percent_observed": valid_na.isin(rule_imputed).mean()[lambda x: x <= 0.5].sort_values()}),
    pd.DataFrame({"percent_observed": valid_na.isin(mice_imputed).mean()[lambda x: x <= 0.5].sort_values()}),
    captions=["validation", "validation with imputed skips", "validation rule imputed", "validation mice imputed"],
)

### create table of highest correlations

In [ ]:
# Visualize correlation matrices
exclude_vars = ["lfd", "fall"]
treat_cat_as_continuous: list[str] = data.meta.loc[lambda x: x["type"].isin(["binary", "ordinal"]), "name"].tolist()

data_excl = data.data.drop(columns=exclude_vars)
corr_assoc_matrix, p_value_matrix = create_corr_assoc_matrix(
    turn_cats_to_continuous(data_excl, treat_cat_as_continuous)
)

corr_assoc_table = (
    pd.DataFrame(
        [
            [vari, varj, corr_assoc, p_value_matrix.loc[vari, varj]]
            for vari, varj in itertools.combinations(data_excl.columns, r=2)
            if (corr_assoc := corr_assoc_matrix.loc[vari, varj]) > 0.9
        ]
    )
    .set_axis(["vari", "varj", "corr_assoc", "p_value"], axis=1)
    .sort_values("corr_assoc", ascending=False)
)
corr_assoc_table

In [ ]:
# Correlations excluding some variables
excluded_vars = [
    # alcohol
    "alk30gr",

    # nicotine
    "hnb30",

    # drugs
    "hasch_12", "ecsta_12", "lsd_12", "heroi_12", "andop_12", "kokai_12", "crack_12", "pilz_12", "nps_12", "amp_12",
    "meth_12",

    # medication
    "m_anr", "t_anr", "anr_30", "m_app", "t_app", "app_30", "m_anab", "t_anab", "anab_30", "schm_30", "m_schm",
    "t_schm", "schl_30", "m_schl", "t_schl", "ber_30", "m_ber", "t_ber", "depr_30", "m_depr", "t_depr", "neur_30",
    "m_neur", "t_neur",
]  # fmt: skip
corr_assoc_table[~corr_assoc_table["vari"].isin(excluded_vars) & ~corr_assoc_table["varj"].isin(excluded_vars)]

### alcohol variables

In [ ]:
# Look at variable group
data.meta[lambda x: x["group"].eq("alcohol")]

In [ ]:
# Ordinal alcohol variable
with pd.option_context("display.max_colwidth", 1000):
    display(data.meta[lambda x: x["name"].eq("alk30kat")])

### nicotine variables

In [ ]:
# Look at variable group
data.meta[lambda x: x["group"].eq("nicotine")]

In [ ]:
# Check if binary variables are the same
contingency_table(data.data["hnb30"], data.data["typ_hnb"], dropna=False, normalize=False)

### medication variables

In [ ]:
# Look at variable group
data.meta[data.meta["group"].eq("medication")]

In [ ]:
# Look at variable group
data.meta[lambda x: x["group"].eq("medication") & ~x["name"].str.contains("_30|^m_|^t_", regex=True)]

### drugs variables

In [ ]:
# Look at variable group
data.meta[lambda x: x["group"].eq("drugs")]

In [ ]:
# Check if categorical variables are the same
vars_12 = [
    "hasch_12", "ecsta_12", "lsd_12", "heroi_12", "andop_12", "kokai_12", "crack_12", "pilz_12", "nps_12", "amp_12",
    "meth_12",
]  # fmt: skip
vars_fr12 = [
    "fr12_ha1", "fr12_ec1", "fr12_ls1", "fr12_he1", "fr12_ao1", "fr12_ko1", "fr12_cr1", "fr12_pi1", "fr12_nps1",
    "fr12_au1", "fr12_meth1",
]  # fmt: skip
display_side_by_side(
    *(
        contingency_table(data.data[vari], data.data[varj], dropna=False, normalize=False)
        for vari, varj in zip(vars_12, vars_fr12)
    )
)

#### general, socioeconomic, and miscellaneous variables

In [ ]:
data.meta[lambda x: ~x["group"].isin(["alcohol", "nicotine", "drugs", "medication"])]